In [1]:
import torch
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from dataclasses import dataclass
from sklearn.metrics import average_precision_score
from catboost import CatBoostClassifier
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_DIR = Path("/kaggle/input/datasets/user_name/gigachat_model")

searching_features = False

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map='auto',
    dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="eager" 
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
)

device = next(model.parameters()).device
print(f"Model on: {device}")

In [3]:
DATA_TRAIN_PATH = 'train_path'
DATA_TEST_PATH = 'test_path'
DATA_VALIDATION_PATH = 'valid_path'

df_train = pd.read_csv(DATA_TRAIN_PATH, encoding='utf-16')
df_test = pd.read_csv(DATA_TEST_PATH, encoding='utf-8')
df_validation = pd.read_csv(DATA_VALIDATION_PATH, encoding='utf-8')

df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
df_validation = df_validation.sample(frac=1, random_state=42).reset_index(drop=True)

# Извлечение фичей

In [4]:
PROBE_LAYERS = [0, 5, 9, 11, 13, 15, 20, 25]

class FeatureAccumulator:
    """Собирает все фичи за один forward pass."""
    def __init__(self, model, probe_layers: list[int] = PROBE_LAYERS):
        self.model = model
        self.probe_layers = probe_layers
        self._hooks = []
        self._hidden = {}

    def attach(self):
        self._hidden.clear()
        for idx in self.probe_layers:
            name = f"layer_{idx}"
            def _make(n):
                def _fn(_, __, out):
                    h = out[0] if isinstance(out, tuple) else out
                    self._hidden[n] = h.detach()
                return _fn
            self._hooks.append(
                self.model.model.layers[idx].register_forward_hook(_make(name))
            )

    def detach(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    def __enter__(self):
        self.attach()
        return self

    def __exit__(self, *_):
        self.detach()

    def compute_features(self, outputs, input_ids, answer_start):
        logits = outputs.logits
        attentions = outputs.attentions
        seq_len = input_ids.shape[1]
        n_answer = seq_len - answer_start

        if n_answer <= 0:
            return self._empty_features()

        answer_logits = logits[0, answer_start-1:seq_len-1, :].float()
        answer_ids = input_ids[0, answer_start:seq_len]
        log_probs = torch.log_softmax(answer_logits, dim=-1)
        token_lp = log_probs.gather(1, answer_ids.unsqueeze(1)).squeeze(-1)
        probs = torch.softmax(answer_logits, dim=-1)
        entropy = -(probs * torch.log(probs+1e-10)).sum(dim=-1)
        top1 = probs.max(dim=-1).values
        top5 = probs.topk(min(5, probs.shape[-1]), dim=-1).values.sum(dim=-1)

        # ---------- 1. Uncertainty (baseline) ----------
        uncertainty_arr = np.array([
            token_lp.mean().item(), token_lp.min().item(), token_lp.max().item(),
            token_lp.std().item() if n_answer>1 else 0.0,
            entropy.mean().item(), entropy.max().item(),
            entropy.std().item() if n_answer>1 else 0.0,
            torch.exp(-token_lp.mean()).item(),
            float(n_answer), token_lp[0].item(),
            top1.mean().item(), top5.mean().item()
        ], dtype=np.float32)

        # ---------- 2. Internal scalars (baseline) ----------
        last_hs = self._hidden[f"layer_{self.probe_layers[-1]}"][0]
        probe_vec = last_hs[answer_start-1].cpu().float().numpy()
        int_scalars = []
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            int_scalars.append(hs[answer_start-1].norm().item())
            int_scalars.append(hs[answer_start:seq_len].norm(dim=-1).mean().item())
            ans_hs = hs[answer_start-1:seq_len-1].unsqueeze(0)
            with torch.no_grad():
                ll = self.model.lm_head(self.model.model.norm(ans_hs)).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ll_e = -(ll_p * torch.log(ll_p+1e-10)).sum(dim=-1)
            int_scalars.append(ll_e.mean().item())
        first_e = int_scalars[2]
        last_e = int_scalars[-1]
        int_scalars.append(first_e - last_e)
        int_scalars.append(last_e/(first_e+1e-10))
        internal_scalar_arr = np.array(int_scalars, dtype=np.float32)

        # ---------- 3. Attention basic ----------
        attention_arr = self._compute_attention(attentions, answer_start, seq_len)

        # ---------- 4. Attention statistics (std, max, mean, sparsity) ----------
        attn_stats_arr = self._compute_attention_stats(attentions, answer_start, seq_len)

        # ---------- 5.Margin_val + semantic_dists + early_conv) ----------
        sota_arr = self._compute_sota_features(answer_logits, answer_ids, seq_len, answer_start)

        # ---------- 6. Layer differences ----------
        layer_diffs_arr = self._compute_layer_differences()

        # ---------- 7. Entropy landscape ----------
        entropy_landscape_arr = self._compute_entropy_landscape(answer_start, seq_len)

        self._hidden.clear()
        return {
            "uncertainty": uncertainty_arr,
            "internal_scalars": internal_scalar_arr,
            "probe_vec": probe_vec,
            "attention": attention_arr,
            "attention_stats": attn_stats_arr,
            "sota": sota_arr,
            "layer_diffs": layer_diffs_arr,
            "entropy_landscape": entropy_landscape_arr,
        }

    # ---- Вспомогательные методы ----
    def _compute_attention(self, attentions, answer_start, seq_len):
        if attentions is None or len(attentions) == 0:
            return np.zeros(3, dtype=np.float32)
        
        last_attn = attentions[-1][0]
        
        if answer_start >= last_attn.shape[1]:
            return np.zeros(3, dtype=np.float32)
        ans_attn = last_attn[:, answer_start:, :]
        
        if ans_attn.shape[1] == 0:
            return np.zeros(3, dtype=np.float32)
    
        attn_entropy = -(ans_attn * torch.log(ans_attn + 1e-10)).sum(dim=-1).mean().item()
        
        prompt_reliance = ans_attn[:, :, :answer_start].sum(dim=-1).mean().item()
        
        max_prompt_focus = ans_attn[:, :, :answer_start].sum(dim=-1).max().item()
        
        return np.array([attn_entropy, prompt_reliance, max_prompt_focus], dtype=np.float32)

    def _compute_attention_stats(self, attentions, answer_start, seq_len):
        if attentions is None:
            return np.zeros(4 * len(self.probe_layers), dtype=np.float32)
            
        stats = []
        for layer_idx in self.probe_layers:
            if layer_idx >= len(attentions):
                stats.extend([0.0, 0.0, 0.0, 0.0])
                continue
                
            attn = attentions[layer_idx][0] # (heads, seq_len, seq_len)
            
            if answer_start >= attn.shape[1]:
                stats.extend([0.0, 0.0, 0.0, 0.0])
                continue
            
            ans_attn = attn[:, answer_start:, :answer_start]
            
            if ans_attn.numel() == 0:
                stats.extend([0.0, 0.0, 0.0, 0.0])
                continue
                
            stats.append(ans_attn.std().item())
            stats.append(ans_attn.max().item())
            stats.append(ans_attn.mean().item())
            stats.append((ans_attn > 0.1).float().mean().item())
            
        return np.array(stats, dtype=np.float32)

    def _compute_sota_features(self, answer_logits, answer_ids, seq_len, answer_start):
        n_answer = len(answer_ids)
        if n_answer == 0:
            n_layers = len(self.probe_layers)
            return np.zeros(1 + n_layers + 1, dtype=np.float32)

        # margin_val
        probs = torch.softmax(answer_logits, dim=-1)
        top2_vals = probs.topk(2, dim=-1).values
        margin_val = (top2_vals[:, 0] - top2_vals[:, 1]).mean().item()

        # layer_semantic_dists
        embeddings = self.model.get_input_embeddings().weight
        layer_semantic_dists = []
        for idx in self.probe_layers:
            hs_full = self._hidden[f"layer_{idx}"][0]
            hs_answer = hs_full[answer_start-1:seq_len-1]
            token_dists = []
            for t_idx in range(min(hs_answer.shape[0], n_answer)):
                h_token = hs_answer[t_idx:t_idx+1].to(self.model.device)
                normed_h = self.model.model.norm(h_token)
                l_logits = self.model.lm_head(normed_h).float()
                t5_indices = l_logits.topk(min(5, l_logits.shape[-1]), dim=-1).indices[0]
                t5_embs = embeddings[t5_indices.to(embeddings.device)].float()
                token_dists.append(torch.cdist(t5_embs, t5_embs).mean().item())
            layer_semantic_dists.append(np.mean(token_dists) if token_dists else 0.0)

        # early_conv
        layer_predictions = []
        for idx in self.probe_layers:
            hs_full = self._hidden[f"layer_{idx}"][0]
            hs_answer = hs_full[answer_start-1:seq_len-1]
            token_preds = []
            for t_idx in range(min(hs_answer.shape[0], n_answer)):
                h_token = hs_answer[t_idx:t_idx+1].to(self.model.device)
                normed_h = self.model.model.norm(h_token)
                l_logits = self.model.lm_head(normed_h).float()
                token_preds.append(l_logits.argmax(dim=-1).cpu())
            if token_preds:
                layer_predictions.append(torch.cat(token_preds))
        if layer_predictions:
            target_ids = answer_ids.cpu()
            min_len = min(len(target_ids), min(len(lp) for lp in layer_predictions))
            target_ids = target_ids[:min_len]
            preds_stack = torch.stack([lp[:min_len] for lp in layer_predictions])
            stability_matrix = (preds_stack == target_ids).float()
            early_conv = stability_matrix.argmax(dim=0).float().mean().item()
        else:
            early_conv = 0.0

        return np.array([margin_val] + layer_semantic_dists + [early_conv], dtype=np.float32)

    def _compute_layer_differences(self):
        diffs = []
        prev_hs = None
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0].cpu()
            if prev_hs is not None:
                prev_hs = prev_hs.cpu() if prev_hs.device != hs.device else prev_hs
                min_len = min(prev_hs.shape[0], hs.shape[0])
                prev_cut = prev_hs[:min_len]
                hs_cut = hs[:min_len]
                cos_sim = torch.nn.functional.cosine_similarity(prev_cut, hs_cut, dim=-1)
                diffs.append((1 - cos_sim).mean().item())
                diffs.append(torch.norm(prev_cut - hs_cut, dim=-1).mean().item())
            prev_hs = hs
        return np.array(diffs, dtype=np.float32)

    def _compute_entropy_landscape(self, answer_start, seq_len):
        entropies = []
        device = next(self.model.lm_head.parameters()).device
        for idx in self.probe_layers:
            hs = self._hidden[f"layer_{idx}"][0]
            if answer_start-1 >= hs.shape[0] or seq_len-1 > hs.shape[0]:
                entropies.append(0.0)
                continue
            ans_hs = hs[answer_start-1:seq_len-1].unsqueeze(0).to(device)
            with torch.no_grad():
                ll = self.model.lm_head(self.model.model.norm(ans_hs)).float()
            ll_p = torch.softmax(ll[0], dim=-1)
            ent = -(ll_p * torch.log(ll_p+1e-10)).sum(dim=-1)
            entropies.append(ent.mean().item())
        if len(entropies) < 2 or all(e == 0 for e in entropies):
            return np.array([0.0, 0.0, 0.0], dtype=np.float32)
        e = np.array(entropies)
        return np.array([e[0]-e[-1], np.gradient(e).std(), np.argmin(e)/len(e)], dtype=np.float32)

    def _empty_features(self):
        n = len(self.probe_layers)
        hidden_size = self.model.config.hidden_size
        return {
            "uncertainty": np.zeros(12, dtype=np.float32),
            "internal_scalars": np.zeros(3*n+2, dtype=np.float32),
            "probe_vec": np.zeros(hidden_size, dtype=np.float32),
            "attention": np.zeros(3, dtype=np.float32),
            "attention_stats": np.zeros(4*n, dtype=np.float32),
            "sota": np.zeros(1+n+1, dtype=np.float32),
            "layer_diffs": np.zeros(2*(n-1) if n>1 else 0, dtype=np.float32),
            "entropy_landscape": np.zeros(3, dtype=np.float32),
        }

In [5]:
def preprocess(tokenizer, prompt, answer):
    messages_prompt = [{"role": "user", "content": prompt}]
    prompt_enc = tokenizer.apply_chat_template(
        messages_prompt,
        add_generation_prompt=True,
        tokenize=True,
    )
    prompt_token_ids = prompt_enc["input_ids"]
    answer_start_idx = len(prompt_token_ids)

    messages_full = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    full_enc = tokenizer.apply_chat_template(
        messages_full,
        tokenize=True,
    )
    token_ids = full_enc["input_ids"]
    token_ids = torch.tensor([token_ids], dtype=torch.long)
    return token_ids, answer_start_idx

In [6]:
def extract_all_features(df, split_name):
    accumulator = FeatureAccumulator(model)
    
    probe_list, unc_list, int_list, attn_list = [], [], [], []
    attn_stats_list, sota_list, layer_diffs_list, entropy_landscape_list = [], [], [], []
    labels = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting {split_name}"):
        token_ids, answer_start_idx = preprocess(tokenizer, row["prompt"], row["model_answer"])
        token_ids = token_ids.to(device)
        
        with accumulator:
            with torch.no_grad():
                outputs = model(token_ids, output_attentions=True)
        
        feats = accumulator.compute_features(outputs, token_ids, answer_start_idx)
        
        probe_list.append(feats["probe_vec"])
        unc_list.append(feats["uncertainty"])
        int_list.append(feats["internal_scalars"])
        attn_list.append(feats["attention"])
        attn_stats_list.append(feats["attention_stats"])
        sota_list.append(feats["sota"])
        layer_diffs_list.append(feats["layer_diffs"])
        entropy_landscape_list.append(feats["entropy_landscape"])
        if "is_hallucination" in row:
            labels.append(row["is_hallucination"])
        else:
            labels.append(0)
        
        del outputs
        torch.cuda.empty_cache()
    
    return {
        "probe_vec_X": np.stack(probe_list).astype(np.float32),
        "uncertainty_X": np.stack(unc_list).astype(np.float32),
        "internal_scalar_X": np.stack(int_list).astype(np.float32),
        "attention_X": np.stack(attn_list).astype(np.float32),
        "attention_stats_X": np.stack(attn_stats_list).astype(np.float32),
        "sota_X": np.stack(sota_list).astype(np.float32),
        "layer_diffs_X": np.stack(layer_diffs_list).astype(np.float32),
        "entropy_landscape_X": np.stack(entropy_landscape_list).astype(np.float32),
        "labels": np.array(labels, dtype=np.int32),
    }


In [ ]:
print("Train:")
#data_train = extract_all_features(df_train, "train")

In [ ]:
print("Test:")
#data_test = extract_all_features(df_test, "test")

In [ ]:
print("Valiation:")
#data_val = extract_all_features(df_validation, "validation")

In [ ]:
#with open('data_train.pkl', 'wb') as f:
#    pickle.dump(data_train, f)

#with open('data_test.pkl', 'wb') as f:
#    pickle.dump(data_test, f)

#with open('data_val.pkl', 'wb') as f:
#    pickle.dump(data_val, f)


In [7]:
# Загружаем уже собранные фичи
with open('features_train_path', 'rb') as file:
    data_train = pickle.load(file)
with open('features_test_path', 'rb') as file:
    data_test = pickle.load(file)
with open('features_valid_path', 'rb') as file:
    data_valid = pickle.load(file)


# Обучение классификатора

In [217]:
class HallucinationClassifier:
    def __init__(self):
        self.pca = PCA(n_components=128, random_state=42)
        self.clf = CatBoostClassifier(
            iterations=2000,         
            learning_rate=0.02,      
            depth=5,                  
            l2_leaf_reg=25,       
            loss_function='Logloss',
            eval_metric='PRAUC',
            bootstrap_type='Bernoulli',
            random_seed=42,
            subsample=0.4,
            colsample_bylevel=0.45,
            min_child_samples=300,
            verbose=100,
            task_type="CPU",
        )
        self.threshold = 0.5
        self.fitted = False
        self.selected_indices = None

    def _prepare_features(self, data, is_train=False):
        p_vec = data["probe_vec_X"] if "probe_vec_X" in data else data["probe_vec"]
        if p_vec.ndim == 1: p_vec = p_vec.reshape(1, -1)
        
        if is_train:
            probe_pca = self.pca.fit_transform(p_vec)
        else:
            probe_pca = self.pca.transform(p_vec)

        parts = [
            probe_pca,
            data.get("internal_scalar_X", data.get("internal_scalars")),
            data.get("uncertainty_X", data.get("uncertainty")),
            data.get("attention_X", data.get("attention")),
            data.get("attention_stats_X", data.get("attention_stats")),
            data.get("sota_X", data.get("sota")),
            data.get("layer_diffs_X", data.get("layer_diffs")),
            data.get("entropy_landscape_X", data.get("entropy_landscape")),
        ]
        
        parts = [p.reshape(len(p), -1) if p.ndim == 1 else p for p in parts]
        return np.hstack(parts)

    def fit(self, data_train, data_val=None):
        y_train = data_train["labels"]
        X_train_full = self._prepare_features(data_train, is_train=True)
        
        eval_set = None
        if data_val is not None:
            X_val_full = self._prepare_features(data_val, is_train=False)
            eval_set = (X_val_full, data_val["labels"])

        print("Starting feature selection...")
        summary = self.clf.select_features(
            X_train_full, y_train,
            eval_set=eval_set,
            features_for_select=list(range(X_train_full.shape[1])),
            num_features_to_select=80,
            algorithm='RecursiveByShapValues',
            steps=3,
            train_final_model=True, 
            plot=False
        )
        
        self.selected_indices = summary['selected_features']
        print(f"Selected {len(self.selected_indices)} features.")
        self.fitted = True

    def predict_proba(self, features_dict):
        if not self.fitted:
            raise RuntimeError("Модель не обучена")
            
        X_full = self._prepare_features(features_dict, is_train=False) 
        probs = self.clf.predict_proba(X_full)
        
        if X_full.shape[0] == 1:
            return float(probs[0, 1])
        return probs[:, 1]

    def predict(self, features_dict):
        prob = self.predict_proba(features_dict)
        label = 1 if prob >= self.threshold else 0
        return label, prob

In [218]:
#Отбираем фичи и обучаем классификатор
clf = HallucinationClassifier()
clf.fit(data_train,data_valid)

Starting feature selection...
Step #1 out of 3
0:	learn: 0.7454269	test: 0.6705979	best: 0.6705979 (0)	total: 15.3ms	remaining: 30.5s
100:	learn: 0.8481613	test: 0.7503086	best: 0.7506888 (89)	total: 897ms	remaining: 16.9s
200:	learn: 0.8744151	test: 0.7628045	best: 0.7629746 (198)	total: 1.73s	remaining: 15.5s
300:	learn: 0.8971588	test: 0.7694837	best: 0.7703536 (294)	total: 2.56s	remaining: 14.4s
400:	learn: 0.9153763	test: 0.7772739	best: 0.7782946 (382)	total: 3.36s	remaining: 13.4s
500:	learn: 0.9330777	test: 0.7835135	best: 0.7841492 (496)	total: 4.17s	remaining: 12.5s
600:	learn: 0.9476644	test: 0.7857471	best: 0.7865541 (555)	total: 4.98s	remaining: 11.6s
700:	learn: 0.9598276	test: 0.7867419	best: 0.7881593 (629)	total: 5.78s	remaining: 10.7s
800:	learn: 0.9686777	test: 0.7864937	best: 0.7881593 (629)	total: 6.65s	remaining: 9.96s
900:	learn: 0.9762194	test: 0.7882085	best: 0.7887102 (876)	total: 7.46s	remaining: 9.1s
1000:	learn: 0.9816881	test: 0.7877720	best: 0.7890899 (92

# Инференс и тесты

In [227]:
@dataclass
class ScoringResult:
    is_hallucination: bool
    is_hallucination_proba: float
    t_model_sec: float = 0.0
    t_overhead_sec: float = 0.0
    t_total_sec: float = 0.0


class GuardianOfTruth:
    def __init__(self, model, tokenizer, classifier: HallucinationClassifier):
        self.model = model
        self.tokenizer = tokenizer
        self.classifier = classifier
        self.accumulator = FeatureAccumulator(model) 

    def score(self, prompt: str, answer: str) -> ScoringResult:
        token_ids, answer_start_idx = preprocess(self.tokenizer, prompt, answer)
        device = next(self.model.parameters()).device
        token_ids = token_ids.to(device)

        t0 = time.perf_counter()
        with self.accumulator:
            with torch.no_grad():
                outputs = model(token_ids, output_attentions=True)
        t_model = time.perf_counter() - t0

        t1 = time.perf_counter()
        features = self.accumulator.compute_features(
            outputs,       
            token_ids,
            answer_start_idx,
        )
        t_overhead = time.perf_counter() - t1

        del outputs
        torch.cuda.empty_cache()

        label, prob = self.classifier.predict(features)

        return ScoringResult(
            is_hallucination=bool(label),
            is_hallucination_proba=prob,
            t_model_sec=t_model,
            t_overhead_sec=t_overhead,
            t_total_sec=t_model + t_overhead,
        )


In [243]:
import time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# Используем уже собранные фичи
X_train_full = clf._prepare_features(data_train, is_train=False)
X_test_full = clf._prepare_features(data_valid, is_train=False)

train_probs = clf.clf.predict_proba(X_train_full)[:, 1]
test_probs = clf.clf.predict_proba(X_test_full)[:, 1]


df_train_res = pd.DataFrame({
    "true": data_train["labels"], 
    "proba": train_probs
})
df_test_res = pd.DataFrame({
    "true": data_valid["labels"], 
    "proba": test_probs
})


overhead_times = []
for i in range(min(100, len(X_test_full))):
    row = X_test_full[i:i+1]
    
    t_start = time.perf_counter()
    _ = clf.clf.predict_proba(row) 
    t_end = time.perf_counter()
    
    overhead_times.append((t_end - t_start) * 1000)

metrics_df = pd.DataFrame([
    {
        'split': "Train", 
        'pr_auc': round(average_precision_score(df_train_res["true"], df_train_res["proba"]), 4)
    },
    {
        'split': "Test",  
        'pr_auc': round(average_precision_score(df_test_res["true"], df_test_res["proba"]), 4)
    },
])
metrics_df = metrics_df.set_index("split")


print(metrics_df.to_string())
print()
print("Timing (test, per sample):")

# forward pass - free
print(f"  model forward : {'N/A':>5} [Фичи уже собраны]") 
print(f"  overhead      : {np.mean(overhead_times):.1f} ms (mean)")
print(f"  overhead p99  : {np.quantile(overhead_times, 0.99):.1f} ms")

overhead_ok = np.quantile(overhead_times, 0.99) < 1000
print(f"  {'PASS' if overhead_ok else 'FAIL'}: p99 overhead < 1000 ms")

       pr_auc
split        
Train  0.9909
Test   0.8275

Timing (test, per sample):
  model forward :   N/A [Фичи уже собраны]
  overhead      : 1.5 ms (mean)
  overhead p99  : 6.5 ms
  PASS: p99 overhead < 1000 ms
